In [1]:
import requests
import math
import pandas as pd
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta

load_dotenv

KEY = os.getenv('API_KEY')
LIST_URL = "https://play.limitlesstcg.com/api/tournaments/"
URL = LIST_URL+"{}"


# Tournaments List

In [7]:
start_date = datetime(2024, 8, 2).date()
end_date = datetime(2024, 8, 9).date()

In [2]:
response = requests.get(LIST_URL, headers={'X-Access-Key':KEY}, params={'game': 'PTCG', 'format': 'STANDARD', 'limit': 1000})

In [8]:
tournament_list = []
id_list = []
date_format = "%Y-%m-%dT%H:%M:%S.%fZ"
i = 0
for entry in response.json():
    # print(entry)
    date = (datetime.strptime(entry['date'], date_format) - timedelta(hours=5)).date()
    if entry['players'] >= 64 and date >= start_date and date <= end_date:
        print(entry['id'], entry['name'] )
        tournament_list.append('{}_{}'.format(i,entry['name']))
        id_list.append(entry['id'])
        i+=1


669a39f3931edf05c6130c24 Ditto Masquerade #10 (100 codes)
66a2b01ba6358f6cc45de7d0 🐉Team Rocket League Day #114
66a2afe7a6358f6cc45de7ce 🐉Team Rocket League Day #113
667470e97e9d4805d0b83265 Late Night Special #8 NA Worlds PREP
6674709d7e9d4805d0b83261 Late Night Special #7 EU WORLDS PREP
66a70c50a6358f6cc45e141d Jefferson Gym Tuesday Night Standard #3
66b10bce82e459668d357342 PokeDeck On-line #65 open  Fábulas Nebulosas
6696baf630c2760605094605 Gray's PokéLeague - Shrouded Fable Special
66b05c0682e459668d35705f Bak & Bak PokeLeague #11
669ef2c8931edf05c6134c35 Free Entry ║ Deck Out Mondays ║ Top 32 400 Codes
66a7ecb782e459668d34f8ef Sunny's Weekly #164 A Fable that is Shrouded
66aa3f1982e459668d350c23 Silver Squad Standard Weekly #1 Trifrost go vroom
669014c430c27606050909ad Redacted Masquerade #10 (100 Codes) Closed Lists
66a69e37a6358f6cc45e1000 Pokémon Battle Park- Shrouded Fable is Legal
66522eec77321005b44fc52d Shrouded Fable (Free Entry)(300 codes)(3hr)
66624dd4e4725805c383dbd2 

# Tournament Data

In [11]:
FILENAME = "data.xlsx"
standings = requests.get(URL.format(id_list[0])+"/standings", headers={'X-Access-Key':KEY})

deck_df = pd.DataFrame(columns=['Player', 'Nation', 'Deck', 'Tournament', 'Placement', 'Day2'])
# standing_df = pd.DataFrame(columns=['Player', 'Wins', 'Losses', 'Ties'])
# pairings_df = pd.DataFrame(columns=['Tour', 'Round', 'Player', 'Opponent', 'Result'])
matchups_df = pd.DataFrame(columns=['Deck', 'Opposing Deck', 'Wins', 'Losses', 'Ties'])

for id, tour in zip(id_list, tournament_list):
    standings = requests.get(URL.format(id)+"/standings", headers={'X-Access-Key':KEY})
    pairings = requests.get(URL.format(id)+"/pairings", headers={'X-Access-Key':KEY})
    rounds = requests.get(URL.format(id)+"/details", headers={'X-Access-Key':KEY}).json()['phases']['phase'==1]['rounds']
    if 'name' not in standings.json()[0]['deck']: continue
    i = 1
    for entry in standings.json():
        name = "{}_{}".format(tour, entry['player'])
        nation = entry['country']
        wins = entry['record']['wins']
        losses = entry['record']['losses']
        ties = entry['record']['ties']
        # deck_df.loc[len(deck_df)] = name, entry['deck']['name']
        if i < 9:
            deck_df.loc[len(deck_df)] = name, nation, entry['deck']['name'], tour, "Top {}".format(pow(2, math.ceil(math.log(i, 2)))), True
        else:
            deck_df.loc[len(deck_df)] = name, nation, entry['deck']['name'], tour, "Out of Top", False
        i+=1

    for entry in pairings.json():
        try:
            # if "freewer" in entry['player1']: print(entry)
            player = "{}_{}".format(tour, entry['player1'])
            opponent = "{}_{}".format(tour, entry['player2'])
            player_deck = deck_df.loc[deck_df['Player'] == player]['Deck'].values[0]
            opp_deck = deck_df.loc[deck_df['Player'] == opponent]['Deck'].values[0]
            matchup = matchups_df.loc[(matchups_df['Deck'] == player_deck) & (matchups_df['Opposing Deck'] == opp_deck)]
            inv_matchup = matchups_df.loc[(matchups_df['Deck'] == opp_deck) & (matchups_df['Opposing Deck'] == player_deck)]
            if entry['winner'] == 0:
                # pairings_df.loc[len(pairings_df)] = tour, entry['round'], player, opponent, 'T'
                # pairings_df.loc[len(pairings_df)] = tour, entry['round'], opponent, player, 'T'
                if len(matchup) == 0:
                    if player_deck != opp_deck: 
                        matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 0, 0, 1
                        matchups_df.loc[len(matchups_df)] = opp_deck, player_deck, 0, 0, 1
                    else:
                        matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 0, 0, 2
                else:
                    matchups_df.loc[matchup.index, 'Ties'] += 1
                    matchups_df.loc[inv_matchup.index, 'Ties'] += 1
            elif entry['player1'] == entry['winner']:
                # pairings_df.loc[len(pairings_df)] = tour, entry['round'], player, opponent, 'W'
                # pairings_df.loc[len(pairings_df)] = tour, entry['round'], opponent, player, 'L'
                if len(matchup) == 0:
                    if player_deck != opp_deck: 
                        matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 1, 0, 0
                        matchups_df.loc[len(matchups_df)] = opp_deck, player_deck, 0, 1, 0
                    else:
                        matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 1, 1, 0
                else:
                    matchups_df.loc[matchup.index, 'Wins'] += 1
                    matchups_df.loc[inv_matchup.index, 'Losses'] += 1
            else:
                # pairings_df.loc[len(pairings_df)] = tour, entry['round'], player, opponent, 'L'
                # pairings_df.loc[len(pairings_df)] = tour, entry['round'], opponent, player, 'W'
                if len(matchup) == 0:
                    if player_deck != opp_deck: 
                        matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 0, 1, 0
                        matchups_df.loc[len(matchups_df)] = opp_deck, player_deck, 1, 0, 0
                    else:
                        matchups_df.loc[len(matchups_df)] = player_deck, opp_deck, 1, 1, 0
                else:
                    matchups_df.loc[matchup.index, 'Losses'] += 1
                    matchups_df.loc[inv_matchup.index, 'Wins'] += 1
        except:
            continue




In [12]:
with pd.ExcelWriter(FILENAME) as writer:
    deck_df.to_excel(writer, sheet_name='decks', index=False)
    # standing_df.to_excel(writer, sheet_name='standings', index=False)
    # pairings_df.to_excel(writer, sheet_name='pairings', index=False)
    matchups_df.to_excel(writer, sheet_name='matchups', index=False)